# POLSCI 3

## Week 12, Notebook Lecture 2: Analyzing experiments with multivariate regression

In the in-class (pdf) lecture and Notebook Lecture 1 this week, I gave you a lot of warnings about using multivariate regression to study causality.

But there is one use of multivariate regression to study causality that is carries less danger: using multivariate regression to analyze data from experiments! 

Once you get the hang of it, this turns out to be even easier than using our old `difference_in_means()` approach, and also allows us to do something else useful, too.

There are two main points in today's lecture.

## **Today's first main point: How to interpret multivariate regressions when analyzing experiments**

The first point is about how to interpret multivariate regressions when you're analyzing experiments.

Here, the concept of a "baseline category" is really important, and I'll show you an example using a new dataset, which we'll also use in the in-class activity.

### Data for today

This dataset is from a pretty neat <a href="http://www.mathiaspoertner.com/uploads/2/6/3/5/26358025/ajps_the_hijab_penalty.pdf">experiment</a> that was conducted by some former graduate students at UC Berkeley (who were students of mine and who also once GSI'd for PS 3 several years ago).

This data set is the result of a large-scale field experiment conducted in 25 cities across Germany, during which 3,797 unknowing bystanders were exposed to brief social encounters with confederates who revealed their ideas regarding gender roles.  

The researchers were interested in how expressed positions on women's rights, as well as ethnic and religious cues, affect social reciprocitity, operationalized here as the willingness of bystanders to help others. 

There were a few aspects to the experiment. We'll cover only one of them in this Notebook Lecture (the part about feminist views) and leave the other one for the in-class activity.

From the authors:

> Our intervention was set up to observe the behavior of unknowing experimental subjects (bystanders) who are exposed to a highly realistic and carefully choreographed sequence of social encounters in public spaces. The intervention followed four steps:
> 1. First, a female confederate approaches a bench at a train station where other individuals are waiting for their train and draws their attention by asking them a question (“Do you know if I can I buy tickets on the train?”). 
> 2. Shortly thereafter, and in the presence of the bystanders, the confederate receives a phone call (from one of the other confederates who was not acting in the specific iteration), and audibly converses with the caller in German...regarding a member of her family (her sister). The conversation is scripted in a manner that reveals the confederate's position on the women's right to choose to pursue a career versus having to stay home to take care of the family. **Note from Professor Dunning: this is the experimental manipulation we will describe below.**
> 3. At the end of the phone call, a bag that the confederate was holding seemingly tears, making her drop a number of lemons, which disperse on the train platform and the confederate appears to be in need of assistance to pick them up. 
> 4. In the final step, team members who were not a part of the intervention record whether each bystander helped the confederate retrieve her lemons. A collage of photographs that capture the key sequences of our experimental intervention are presented in Figure 1.
> 
> ![](scene.jpg)
> 
> _Figure 1: Unknowing bystanders watch and listen as the confederate takes a call and conducts a conversation with a friend (a), in the process revealing her attitudes toward the role of women in society (family and work). Following the phone call, the confederate drops her possessions (lemons), which disperse on the platform (b). We observe whether bystanders assist the confederate in collecting her possessions (c)._

Here were the contents of the phone calls:

> - In the "regressive" gender attitude condition, the confederate expresses disappointment with her sister, who has decided to get a job rather than stay at home and take care of her husband and kids. The confederate states that she believes her role as a woman is to stay at home and take care of her family.
> - In the "progressive" attitude condition, the confederate expresses her approval of her sister’s decision to get a job rather than stay home and take care of her husband and kids. She states that she believes that women should not sacrifice their careers to stay at home and take care of their family.
> - In the neutral control condition, the confederate has a conversation of roughly equal length about an innocuous matter unrelated to her attitudes regarding women and of no sociopolitical valence.
>
> The specific issue of women’s career advancement was chosen because it has been a crucial concern of the women’s rights movement in Germany; most—but not all—native women hold progressive views.

The authors also randomized whether the women were wearing hijabs (head scarves), which provide a religious and ethnic cue in the German context. We'll ignore this for now -- we'll look at that during the in-class activity.

Let's take a look at the data:

In [1]:
library(estimatr)

data <- read.csv("ps3_phonecall_clean_lecture.csv")
head(data)

,anyhelp,genderatt,bystander,femprop
,<int>,<chr>,<int>,<dbl>
1,1,Progressive,3,0.6666667
2,1,Neutral,2,0.5000000
3,0,Neutral,2,0.5000000
4,1,Conservative,1,1.0000000
5,1,Progressive,3,0.3333333
6,1,Progressive,2,1.0000000


(This is a reduced version of the dataset with fewer columns; we'll look at a fuller version in the in-class activity).

Here, each row in the dataset is one interaction where a person dropped their lemons at a German bus stop after talking on the phone.

Here is a quick rundown of what each column means:

- `anayhelp`: **outcome variable**: whether any bystander offered help
- `genderatt`: **treatment variable/experimental manipulation**: "progresive," "regressive," or "neutral" gender attitude (Progressive, Neutral, Regressive) as expressed by the confederate talking on the phone.
- `bystander`: number of bystanders 
- `femprop`: proportion of female bystanders

### Using regression to analyze experiments

Here's the values of that treatment variable again:

In [2]:
table(data$genderatt)


Conservative      Neutral  Progressive 
         618          980          597 

Now, what happens if we first run a **bivariate regression**, predicting the outcome (whether people get help) with that treatment variable?

In [3]:
summary(lm(anyhelp ~ genderatt, data))


Call:
lm(formula = anyhelp ~ genderatt, data = data)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.7388 -0.7006  0.2612  0.2630  0.2994 

Coefficients:
                     Estimate Std. Error t value Pr(>|t|)    
(Intercept)           0.70065    0.01791  39.124   <2e-16 ***
genderattNeutral      0.03813    0.02287   1.667   0.0956 .  
genderattProgressive  0.03637    0.02555   1.424   0.1547    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.4452 on 2192 degrees of freedom
Multiple R-squared:  0.001435,	Adjusted R-squared:  0.0005238 
F-statistic: 1.575 on 2 and 2192 DF,  p-value: 0.2073


Notice that the treatment has three values but the regression is only showing you two coefficients. 

Why? Because **a regression on experimental data shows you the _difference_ between each of the treatment conditions and a baseline**. The baseline is the value of the treatment that isn't shown (we call this the "omitted category"). In this case, the Conservative condition isn't shown, so we know that the regression is showing us a) the effect of the Neutral condition _relative to the Conservative condition_ and the effect of the Progressive condition _relative to the Conservative condition_.

Moreover, the standard error, t-value, and p-value columns mean exactly what they meant before! (Unfortunately, the regression output doesn't automatically include a confidence interval, you would have to calculate it by hand by using $\text{Confidence Interval} = \text{Estimate} +/- 1.96 * \text{Standard Error}$.)

In contrast to regression analysis of observational data, we can more readily trust the standard errors, *t*-statistics, and *p*-values with experimental data.  

After all, in an experiment the researcher controls the introduction of random error---by randomly assignment subjects to treatment groups, or, equivalently, generating simple random samples from the experimental study group and applying different treatments to each sample.  Any deviations from simple random sampling are also accurately understood as features of the design, and this makes it easier to adjust for such features in the analysis. (We won't discuss such adjustments further in this class, however). 

So that makes it more reliable to use the laws of probability and statistics to estimate parameters like the average causal effect, as well as the sampling distribution of the estimators.

Now, let's now relate the bivariate regression results to how we have analyzed experiments so far in the class.

In the language of `difference_in_means()`, the category that the regression doesn't show is the same as `condition1` (the baseline), and it can show us the difference between that condition and multiple others on each line.

Let's verify this is true by looking at this our old, old (pre-`difference_in_means`) way.

In [4]:
cons.call <- subset(data, genderatt == 'Conservative')
cons.mean <- mean(cons.call$anyhelp)
cons.mean

[1] 0.7006472

In [5]:
neutral.call <- subset(data, genderatt == 'Neutral')
neutral.mean <- mean(neutral.call$anyhelp)
neutral.mean

[1] 0.7387755

In [6]:
progressive.call <- subset(data, genderatt == 'Progressive')
progressive.mean <- mean(progressive.call$anyhelp)
progressive.mean

[1] 0.7370184

Do we see the same estimates as we did in the regression? Yes:

In [7]:
neutral.mean - cons.mean

[1] 0.03812826

In [8]:
progressive.mean - cons.mean

[1] 0.03637118

This is also identical to what we were doing with `difference_in_means()` before:

In [9]:
difference_in_means(anyhelp ~ genderatt, data,
                    condition1 = 'Conservative', condition2 = 'Neutral')

Design:  Standard 
                   Estimate Std. Error  t value  Pr(>|t|)     CI Lower
genderattNeutral 0.03812826  0.0231746 1.645261 0.1001634 -0.007336428
                   CI Upper       DF
genderattNeutral 0.08359295 1270.754

In [10]:
difference_in_means(anyhelp ~ genderatt, data,
                    condition1 = 'Conservative', condition2 = 'Progressive')

Design:  Standard 
                       Estimate Std. Error  t value  Pr(>|t|)    CI Lower
genderattProgressive 0.03637118 0.02579034 1.410264 0.1587182 -0.01422744
                       CI Upper       DF
genderattProgressive 0.08696979 1212.972

### **Takeaways**: 

1. When you run a bivariate regression like `lm(outcome ~ treatment, data)`, **you are comparing an *omitted category* of the treatment to all of the remaining categories,** one-by-one. 

2. As long as there are no other variables in the regression except for the treatment variables (i.e., it is a bivariate regression), the **`(Intercept)` gives you the mean of the outcome in this omitted (baseline) group**. 

3. The **coefficients on each treatment** tell you the **difference between the mean in that omitted group and the mean in each of the other experimental groups**---just as `difference_in_means()` does.

------

## **Today's second main point: When analyzing experimental data, control variables in regressions serve a different purpose: decreasing noise, not bias**

We learned that experiments have no omitted variable bias, but they do have noise. This is why we have standard errors and $p$-values, to quantify how much noise there is, and whether our estimate might have arisen by chance.

We also learned that, when analyzing observational data, the purpose of control variables is to try to reduce omitted variable bias (although it's unlikely this will ever fully succeed).

Today, I want to show you that _when analyzing experimental data, we can use control variables to reduce noise_!

How useful control variables will be depends on _how predictive the variables are of the outcome_ (that is, how "prognostic" they are). If a control variable predicts the outcome well, it gets rid of noise in the outcome that can't be due to the effect of the treatment. This leaves less noise left over.

Important note for your final projects: We cannot include anything as a covariate that treatment might itself affect, because this might explain some of the variation in the outcome that is due to the treatment. Though this point is beyond our focus in this course, including such "post-treatment" variables in your regression can actually lead to bias in your estimate of the effect of the treatment itself.

To be safe **only include as covariates variables that were measured prior to the treatment being implemented**.

We do this just like before, using the `+` sign to add additional variables.

We can start with the same regression as before:

In [11]:
summary(lm(anyhelp ~ genderatt, data)) # same regression as before


Call:
lm(formula = anyhelp ~ genderatt, data = data)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.7388 -0.7006  0.2612  0.2630  0.2994 

Coefficients:
                     Estimate Std. Error t value Pr(>|t|)    
(Intercept)           0.70065    0.01791  39.124   <2e-16 ***
genderattNeutral      0.03813    0.02287   1.667   0.0956 .  
genderattProgressive  0.03637    0.02555   1.424   0.1547    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.4452 on 2192 degrees of freedom
Multiple R-squared:  0.001435,	Adjusted R-squared:  0.0005238 
F-statistic: 1.575 on 2 and 2192 DF,  p-value: 0.2073


Now let's add control variables:

In [12]:
summary(lm(anyhelp ~ genderatt + bystander + femprop, data))


Call:
lm(formula = anyhelp ~ genderatt + bystander + femprop, data = data)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.7575 -0.6850  0.2603  0.2787  0.3485 

Coefficients:
                       Estimate Std. Error t value Pr(>|t|)    
(Intercept)           0.7172187  0.0287499  24.947   <2e-16 ***
genderattNeutral      0.0381300  0.0228971   1.665    0.096 .  
genderattProgressive  0.0363211  0.0255780   1.420    0.156    
bystander             0.0004274  0.0092799   0.046    0.963    
femprop              -0.0330562  0.0242357  -1.364    0.173    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.4452 on 2190 degrees of freedom
Multiple R-squared:  0.002283,	Adjusted R-squared:  0.0004603 
F-statistic: 1.253 on 4 and 2190 DF,  p-value: 0.2866


Note that the number of bystanders and the proportion of them that are female can be consider "pre-treatment" variables.  They have the value they have in any given setting; and then the message spoken by the confederate on the phone is randomly determined, independent of the value of these variables.  We do not expect that the number of bystanders or the proportion female would be affected in this case by the experimental treatment that is assigned.

In this case, we can see that the covariates aren't very predictive of the outcome (they aren't even related in a statistically significant manner) and so the precision doesn't increase very much (in fact, not at all). However, there's no harm in including them.

Notice something else as well.  The coefficients on the experimental treatments (genderattNeutral and genderattProgressive) barely change, relative to the bivariate regression.  The reason is that treatment assignment is randomized---and therefore we expect the treatment variables to be balanced across values of the pre-treatment covariates.  This means treatment assignment is independent of the covariates.  Although there might be a minor correlation in the data, due to chance variation, it should not be very important.

If you think back to the more technical material about what a multivariate regression does, in the last part of Notebook Lecture 1, it might therefore make sense that the coefficients don't change.  We learned there that you can get the coefficient on a variable in a multiple regression in another way, other than just running the multivariate regression using `lm()`.  First, you regress the treatment on all the other variables on the right-hand side of the multivariate regression.  Then, you regress the outcome variable on the residuals from this first regression.  But notice that if the treatment variable is independent of the other covariates, regressing the outcome on the residuals of the first regression is about the same as simply a bivariate regression of the outcome on the variable itself. After all, the covariates do not predict treatment assignment (we expect them to be independent of treatment assignment).  That is why the coefficients on the treatment variable in the multivariate regression are about the same as in the bivariate regression.

### **Takeaways**:
- **We can include both randomized treatments and non-randomized covariates in a regression.** This is an optional enhancement that can help increase precision.
- **Don't include covariates that were measured after treatment,** especially if they could plausibly themselves be affected by the treatment. Thhis could get rid of variation in the outcome related to the effect of the treatment's effects. We don't want that -- we only want to get rid of variation that couldn't possibly be due to the treatment (because it was measured prior).
- **We still need to interpret the estimates (or "coefficients") on covariates (as opposed to treatments) as pure correlations and not as causal.** Just because _a variable_ was randomized (the treatment) doesn't mean _every_ variable can now be interpreted as causal. For example, in this experiment, `genderatt` was randomized, but `femprop` wasn't! So, we can interpret the coefficients on the `genderatt` variables as causal, but not on the others (like `femprop`).